In [1]:
import os
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import optuna
import warnings
import logging

warnings.filterwarnings('ignore', category=FutureWarning)
optuna.logging.set_verbosity(logging.WARNING)

print("Loading and filtering data...")
data_dir = '../preprocesing_data/processed_csv'
sales_path = os.path.join(data_dir, 'sales_data_preprocessed.csv')
weather_path = os.path.join(data_dir, 'weather_data_hourly.csv')
holidays_path = os.path.join(data_dir, 'holidays_data_preprocessed.csv')
events_path = os.path.join(data_dir, 'aberdeen_events_master_timeline.csv')

# ==========================================
# 1. Sales (GLOBAL MELT)
# ==========================================
sales_df = pd.read_csv(sales_path)
sales_df['Date'] = pd.to_datetime(sales_df['Date']).dt.normalize()
if 'Time' in sales_df.columns:
    sales_df = sales_df.drop(columns=['Time'])
daily_sales = sales_df.groupby('Date').sum(numeric_only=True).reset_index()

product_cols = [c for c in daily_sales.columns if c != 'Date']

# MELT: wide -> long
df_long = pd.melt(daily_sales, id_vars=['Date'], value_vars=product_cols,
                  var_name='Product_Name', value_name='Sales')
df_long['Product_Name'] = df_long['Product_Name'].astype('category')

# Only keep my 74 target products

# ==========================================
# PRODUCTS TO FORECAST
# ==========================================
# These are the 74 specific menu items I want to predict
PRODUCTS_TO_FORECAST = [
 'Avo & Hal Muffin',
 'Avo, Egg & Bacon',
 'Avo, Feta & Tom',
 'Avocado on Toast',
 'Bacon',
 'Bacon Bap',
 'Bacon Egg Brioch',
 'Bacon Waffle',
 'Baked Beans',
 'Baked Beans JP',
 'Bean Soldiers',
 'Big Breakfast',
 'Black Pudding',
 'Breakfast Hash',
 'Breakfast Muffin',
 'Breakfast Wrap',
 'Buttd Mushrooms',
 'Cheese & Bean JP',
 'Cheese JP',
 'Chick Flatbread',
 'Chicken Club',
 'Chilli Carne JP',
 'Egg Bacon Brioch',
 'Egg Bap',
 'Extra Beans',
 'F.Eggs on Toast',
 'Festive Stack',
 'Fried Egg',
 'Hash Brown',
 'Hash Brown Bites',
 'Little Avo Toast',
 'Little Bean Toas',
 'Little Egg Toast',
 'Ltle Bfast Bacon',
 'Ltle Bfast Saus',
 'Mini Hash Browns',
 'P.Eggs on Toast',
 'Poached Egg',
 'Posh Beans',
 'Roll & Butter ',
 'S.Eggs on Toast',
 'Sausage',
 'Sausage Bap',
 'Scrambled Egg',
 'Streaky Bacon',
 'Tattie Scone',
 'The Breakfast',
 'Toasted Teacake',
 'Tuna JP',
 'Tuna Mayo Mix',
 'Tuna Melt Panini',
 'Tuna Panini',
 'Tuna Toastie',
 'Veg Sausage Bap',
 'Vegan Breakfast',
 'Vegan Sausage',
 'Veggie Bap',
 'Veggie Breakfast',
 'Bakery',
 'White Toast Bread',
 'Brown Toast Bread',
 'Porridge',
 'Sourdough Toast Bread',
 'Multiseed Toast Bread',
]
df_long = df_long[df_long['Product_Name'].isin(PRODUCTS_TO_FORECAST)].reset_index(drop=True)

# ==========================================
# 2. Weather
# ==========================================
weather_df = pd.read_csv(weather_path)
weather_df['Date'] = pd.to_datetime(weather_df['Date']).dt.normalize()

weather_agg = {
    'apparent_temperature': 'mean',
    'precipitation': 'sum',
    'snowfall': 'sum',
    'snow_depth': 'max',
    'relative_humidity_2m': 'mean',
    'cloud_cover': 'mean',
    'visibility': 'mean',
    'wind_speed_10m': 'mean',
    'wind_gusts_10m': 'max'
}

if 'weather_code' in weather_df.columns:
    weather_df['is_clear'] = (weather_df['weather_code'] == 0).astype(int)
    weather_df['is_cloudy'] = weather_df['weather_code'].isin([1, 2, 3, 45, 48]).astype(int)
    weather_df['is_rain'] = weather_df['weather_code'].isin(
        [51, 53, 55, 56, 57, 61, 63, 65, 66, 67, 80, 81, 82, 95, 96, 99]).astype(int)
    weather_df['is_snow'] = weather_df['weather_code'].isin([71, 73, 75, 77, 85, 86]).astype(int)
    weather_agg.update({'is_clear': 'max', 'is_cloudy': 'max', 'is_rain': 'max', 'is_snow': 'max'})

daily_weather = weather_df.groupby('Date').agg(weather_agg).reset_index()
if 'Time' in daily_weather.columns:
    daily_weather = daily_weather.drop(columns=['Time'])

# ==========================================
# 3. Holidays & Events
# ==========================================
holidays_df = pd.read_csv(holidays_path)
holidays_df['Date'] = pd.to_datetime(holidays_df['Date']).dt.normalize()
daily_holidays = holidays_df.groupby('Date').max().reset_index()

# Holiday proximity
daily_holidays['is_holiday_lag_1'] = daily_holidays['is_holiday'].shift(1).fillna(0)
daily_holidays['is_holiday_lead_1'] = daily_holidays['is_holiday'].shift(-1).fillna(0)

events_df = pd.read_csv(events_path)
events_df['Date'] = pd.to_datetime(events_df['Date']).dt.normalize()
daily_events = events_df.groupby('Date').max(numeric_only=True).reset_index()


/home/alex/miniconda3/envs/jupyterproject-project/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading and filtering data...


In [2]:
# ==========================================
# 4. Merge & Feature Engineering
# ==========================================
# Time features — cyclical + raw day_of_week for tree splitting
df_long['day_of_week'] = df_long['Date'].dt.dayofweek
df_long['day_sin'] = np.sin(2 * np.pi * df_long['day_of_week'] / 7)
df_long['day_cos'] = np.cos(2 * np.pi * df_long['day_of_week'] / 7)
df_long['month'] = df_long['Date'].dt.month
df_long['month_sin'] = np.sin(2 * np.pi * (df_long['month'] - 1) / 12)
df_long['month_cos'] = np.cos(2 * np.pi * (df_long['month'] - 1) / 12)
df_long['day_of_month'] = df_long['Date'].dt.day
df_long['Is_Weekend'] = df_long['Date'].dt.dayofweek.isin([5, 6]).astype(int)
df_long['Year'] = df_long['Date'].dt.year
df_long['week_of_year'] = df_long['Date'].dt.isocalendar().week.astype(int)

df = df_long.merge(daily_weather, on='Date', how='left')
df = df.merge(daily_holidays, on='Date', how='left')
df = df.merge(daily_events, on='Date', how='left')

# Drop problematic object columns
for col in ['date', 'Time']:
    if col in df.columns:
        df = df.drop(columns=[col])

# Fill NaN for non-categorical columns only
categorical_cols = df.select_dtypes(include=['category']).columns
non_categorical_cols = df.columns.difference(categorical_cols)
df[non_categorical_cols] = df[non_categorical_cols].fillna(0)

# --- Lag features (grouped by product) ---
df = df.sort_values(['Product_Name', 'Date']).reset_index(drop=True)

for lag in [1, 2, 7, 14, 30]:
    df[f'sales_{lag}_step_ago'] = df.groupby('Product_Name', observed=False)['Sales'].shift(lag)

# --- Rolling features on 1-step lag (mean and std only — dropping max to reduce noise) ---
for w in [3, 7, 14]:
    df[f'rolling_{w}d_avg'] = df.groupby('Product_Name', observed=False)['sales_1_step_ago'].transform(
        lambda x: x.rolling(w, min_periods=1).mean()
    )
    df[f'rolling_{w}d_std'] = df.groupby('Product_Name', observed=False)['sales_1_step_ago'].transform(
        lambda x: x.rolling(w, min_periods=1).std()
    ).fillna(0)

# Sales momentum
df['sales_momentum'] = df['sales_1_step_ago'] - df['sales_7_step_ago']

# Expanding mean per product (stable long-term baseline — no leakage since it uses shift)
df['expanding_mean'] = df.groupby('Product_Name', observed=False)['sales_1_step_ago'].transform(
    lambda x: x.expanding(min_periods=1).mean()
)

# Ratio: yesterday vs 7-day average (captures whether yesterday was unusually high/low)
df['ratio_1d_vs_7d'] = df['sales_1_step_ago'] / (df['rolling_7d_avg'] + 1e-8)

df = df.dropna().reset_index(drop=True)
df['Sales'] = df['Sales'].clip(lower=0)

# REMOVED: Total_Daily_Sales and total_sales_1_step_ago (DATA LEAKAGE)
# These cannot be known at prediction time during recursive forecasting
# unless all products are predicted simultaneously and summed.

feature_cols = [c for c in df.columns if c not in ['Date', 'Sales']]

print(f"Features ({len(feature_cols)}): {feature_cols}")
print(f"Products: {df['Product_Name'].cat.categories.tolist()}")
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")


Features (47): ['Product_Name', 'day_of_week', 'day_sin', 'day_cos', 'month', 'month_sin', 'month_cos', 'day_of_month', 'Is_Weekend', 'Year', 'week_of_year', 'apparent_temperature', 'precipitation', 'snowfall', 'snow_depth', 'relative_humidity_2m', 'cloud_cover', 'visibility', 'wind_speed_10m', 'wind_gusts_10m', 'is_clear', 'is_cloudy', 'is_rain', 'is_snow', 'is_holiday', 'holiday_importance', 'Days_Until_Next_Holiday', 'is_holiday_lag_1', 'is_holiday_lead_1', 'is_festival', 'festival_importance', 'is_match_day', 'match_importance', 'sales_1_step_ago', 'sales_2_step_ago', 'sales_7_step_ago', 'sales_14_step_ago', 'sales_30_step_ago', 'rolling_3d_avg', 'rolling_3d_std', 'rolling_7d_avg', 'rolling_7d_std', 'rolling_14d_avg', 'rolling_14d_std', 'sales_momentum', 'expanding_mean', 'ratio_1d_vs_7d']
Products: ['Almd & Aprct Bar', 'Apl & Blk Smthie', 'Avo & Hal Muffin', 'Avo, Egg & Bacon', 'Avo, Feta & Tom', 'Avocado on Toast', 'BD Curry Roll', 'BLT', 'Bacon', 'Bacon Bap', 'Bacon Egg Brioch',

In [3]:
# ==========================================
# 5. SPLIT & TUNE (GLOBAL)
# ==========================================
train_end = pd.to_datetime('2025-10-01')
val_end = pd.to_datetime('2025-11-01')
test_end = pd.to_datetime('2025-11-30')

train_df = df[df['Date'] <= train_end].copy()
val_df = df[(df['Date'] > train_end) & (df['Date'] <= val_end)].copy()
test_df = df[(df['Date'] > val_end) & (df['Date'] <= test_end)].copy().reset_index(drop=True)

X_train, y_train = train_df[feature_cols], train_df['Sales']
X_val, y_val = val_df[feature_cols], val_df['Sales']

print(f"\nTrain: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print("\nRunning Optuna Hyperparameter Tuning (75 trials)...")


def objective(trial):
    param = {
        "n_estimators": 1500,
        "early_stopping_rounds": 50,
        "learning_rate": trial.suggest_float("learning_rate", 5e-3, 0.1, log=True),
        "max_depth": trial.suggest_int("max_depth", 4, 8),
        "min_child_weight": trial.suggest_int("min_child_weight", 2, 8),
        "subsample": trial.suggest_float("subsample", 0.6, 0.95),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.9),
        "gamma": trial.suggest_float("gamma", 1e-4, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.01, 5.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.01, 5.0, log=True),
        "enable_categorical": True,
        "tree_method": "hist"
    }

    model = xgb.XGBRegressor(**param)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    return rmse


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)
best_params = study.best_params
best_params.update({
    "n_estimators": 1500,
    "early_stopping_rounds": 30,
    "enable_categorical": True,
    "tree_method": "hist"
})

print(f"Best RMSE: {study.best_value:.4f}")
print(f"Best params: {best_params}")

print("\nTraining final global model...")
model = xgb.XGBRegressor(**best_params)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)



Train: 85760, Val: 1984, Test: 1856

Running Optuna Hyperparameter Tuning (75 trials)...
Best RMSE: 2.7140
Best params: {'learning_rate': 0.01766872233012172, 'max_depth': 8, 'min_child_weight': 6, 'subsample': 0.7433200654326437, 'colsample_bytree': 0.8246135833985084, 'gamma': 0.02621216297625542, 'reg_lambda': 2.6046985297465084, 'reg_alpha': 0.06865821943029198, 'n_estimators': 1500, 'early_stopping_rounds': 30, 'enable_categorical': True, 'tree_method': 'hist'}

Training final global model...


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8246135833985084, device=None,
             early_stopping_rounds=30, enable_categorical=True,
             eval_metric=None, feature_types=None, gamma=0.02621216297625542,
             grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.01766872233012172,
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=8, max_leaves=None,
             min_child_weight=6, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=1500, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

In [4]:
# ==========================================
# 6. RECURSIVE FORECAST — ALL PRODUCTS
# ==========================================
print("\nRunning Recursive Forecast for ALL Products...")

# Dynamic features that need rebuilding each step
LAG_MAP = {1: 'sales_1_step_ago', 2: 'sales_2_step_ago', 7: 'sales_7_step_ago',
           14: 'sales_14_step_ago', 30: 'sales_30_step_ago'}
ROLLING_WINDOWS = [3, 7, 14]

all_products = test_df['Product_Name'].cat.categories.tolist()
test_dates = sorted(test_df['Date'].unique())

# Pre-index test_df for fast access
test_df['XGB_Prediction'] = 0.0
test_df['original_sales_1_step_ago'] = test_df['sales_1_step_ago'].copy()

# Build lookup: (product, date) -> row index in test_df
idx_lookup = {}
for idx, row in test_df.iterrows():
    idx_lookup[(row['Product_Name'], row['Date'])] = idx

# Process day by day, all products at once
for day_i, current_date in enumerate(test_dates):
    # Get all product rows for this date
    day_mask = test_df['Date'] == current_date
    day_indices = test_df.index[day_mask].tolist()

    if len(day_indices) == 0:
        continue

    # Predict all products for this day
    X_day = test_df.loc[day_indices, feature_cols]
    preds = model.predict(X_day)
    preds = np.clip(preds, 0, None)

    # Store predictions
    test_df.loc[day_indices, 'XGB_Prediction'] = preds

    # Feed predictions forward into future lag columns
    for row_idx, pred_val in zip(day_indices, preds):
        product = test_df.at[row_idx, 'Product_Name']

        for lag_days, lag_col in LAG_MAP.items():
            future_date = current_date + pd.Timedelta(days=lag_days)
            future_key = (product, future_date)
            if future_key in idx_lookup:
                future_idx = idx_lookup[future_key]
                test_df.at[future_idx, lag_col] = pred_val

    # Rebuild derived features for TOMORROW (next date in test set)
    if day_i + 1 < len(test_dates):
        next_date = test_dates[day_i + 1]
        next_mask = test_df['Date'] == next_date
        next_indices = test_df.index[next_mask].tolist()

        for next_idx in next_indices:
            product = test_df.at[next_idx, 'Product_Name']

            # Get the history of sales_1_step_ago for this product up to tomorrow
            product_mask = test_df['Product_Name'] == product
            product_df = test_df.loc[product_mask].sort_values('Date')

            # Position of next_date row within this product's rows
            product_dates = product_df['Date'].values
            pos = np.searchsorted(product_dates, np.datetime64(next_date))

            # Rebuild rolling stats using iloc for safety
            lag1_history = product_df['sales_1_step_ago'].values

            for w in ROLLING_WINDOWS:
                start = max(0, pos - (w - 1))
                window = lag1_history[start:pos + 1]
                test_df.at[next_idx, f'rolling_{w}d_avg'] = np.mean(window)
                test_df.at[next_idx, f'rolling_{w}d_std'] = np.std(window, ddof=1) if len(window) > 1 else 0.0
                # rolling_max removed

            # Rebuild momentum
            test_df.at[next_idx, 'sales_momentum'] = (
                test_df.at[next_idx, 'sales_1_step_ago'] -
                test_df.at[next_idx, 'sales_7_step_ago']
            )

            # Rebuild expanding mean
            if pos > 0:
                test_df.at[next_idx, 'expanding_mean'] = np.mean(lag1_history[:pos + 1])

            # Rebuild ratio
            rolling_7d = test_df.at[next_idx, 'rolling_7d_avg']
            test_df.at[next_idx, 'ratio_1d_vs_7d'] = (
                test_df.at[next_idx, 'sales_1_step_ago'] / (rolling_7d + 1e-8)
            )

print("Recursive forecast complete for all products.")



Running Recursive Forecast for ALL Products...
Recursive forecast complete for all products.


In [5]:
# ==========================================
# 7. SCORECARD (PER-PRODUCT & AGGREGATE)
# ==========================================
# ==========================================
# NORTH STAR METRIC FUNCTION
# ==========================================
# I'm using a tiered metric system here so different stakeholders get the right view:
#   - WAPE:  Best global metric — tells the business "what % of total inventory did we get wrong?"
#            Unlike MAPE, WAPE weights by volume so high-sellers matter more than low-sellers.
#   - MASE:  Best "is the ML working?" metric — if MASE < 1.0, my model beats naive lag-1 baseline.
#            If MASE > 1.0, I'd be better off just using yesterday's sales as my forecast.
#   - MAE:   Best ground-level metric — kitchen staff care about units, not percentages.
#   - Bias:  Directional error — negative means I'm under-predicting (running out of stock),
#            positive means I'm over-predicting (wasting food).

def calculate_north_star_metrics(evaluation_slice):
    """Calculate my North Star metrics for a given time horizon slice.

    Returns a dict with WAPE, MASE, MAE, Bias (and legacy metrics for backwards compatibility).
    These are the metrics I actually care about for decision-making:
      - WAPE for stakeholder reporting (volume-weighted accuracy)
      - MASE for model validation (am I beating a naive baseline?)
      - MAE + Bias for kitchen operations (units off, direction of error)
    """
    if len(evaluation_slice) == 0:
        return {m: 0 for m in ['WAPE', 'MASE', 'MAE', 'Bias', 'MSE', 'RMSE', 'MAPE', 'SMAPE']}

    actual_sales = evaluation_slice['Sales'].values.astype(float)
    predicted_sales = evaluation_slice['XGB_Prediction'].values.astype(float)
    # Use 'sales_1_step_ago' to compare against the real past sales
    naive_baseline = evaluation_slice['sales_1_step_ago'].values.astype(float)

    # --- NORTH STAR 1: WAPE (Weighted Absolute Percentage Error) ---
    # Sum of absolute errors / Sum of actual sales
    # This tells me the total % of inventory I got wrong across all products
    total_absolute_errors = np.sum(np.abs(actual_sales - predicted_sales))
    total_actual_sales = np.sum(np.abs(actual_sales))
    wape = total_absolute_errors / total_actual_sales if total_actual_sales > 0 else np.nan

    # --- NORTH STAR 2: MASE (Mean Absolute Scaled Error) ---
    # My model's MAE divided by naive baseline MAE
    # < 1.0 means my model adds value, > 1.0 means I should just use yesterday's number
    mae = mean_absolute_error(actual_sales, predicted_sales)
    naive_baseline_mae = mean_absolute_error(actual_sales, naive_baseline)
    mase = mae / naive_baseline_mae if naive_baseline_mae > 0 else np.nan

    # --- NORTH STAR 3: MAE (Mean Absolute Error in units) ---
    # The kitchen needs to know: "how many units off will I typically be?"
    # (mae already calculated above)

    # --- NORTH STAR 4: Bias (directional error) ---
    # Negative = I'm under-predicting (we'll run out of stock)
    # Positive = I'm over-predicting (we'll waste food)
    forecast_bias = np.mean(predicted_sales - actual_sales)

    # --- Legacy metrics (keeping for backwards compatibility) ---
    mse = mean_squared_error(actual_sales, predicted_sales)
    rmse = np.sqrt(mse)

    # MAPE — only on non-zero actuals to avoid division by zero
    nonzero_mask = actual_sales > 0
    mape = mean_absolute_percentage_error(actual_sales[nonzero_mask], predicted_sales[nonzero_mask]) if nonzero_mask.sum() > 0 else np.nan

    # SMAPE — symmetric version, less sensitive to zeros
    abs_errors = np.abs(predicted_sales - actual_sales)
    abs_denominator = (np.abs(actual_sales) + np.abs(predicted_sales)) / 2.0
    valid_mask = abs_denominator > 0
    smape = np.mean(abs_errors[valid_mask] / abs_denominator[valid_mask]) if valid_mask.sum() > 0 else np.nan

    return {'WAPE': wape, 'MASE': mase, 'MAE': mae, 'Bias': forecast_bias,
            'MSE': mse, 'RMSE': rmse, 'MAPE': mape, 'SMAPE': smape}


    y_true = df_slice['Sales'].values.astype(float)
    y_pred = df_slice['XGB_Prediction'].values.astype(float)
    y_naive = df_slice['original_sales_1_step_ago'].values.astype(float)

    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)

    # Guard against zero sales for MAPE
    nonzero_mask = y_true > 0
    if nonzero_mask.sum() > 0:
        mape = mean_absolute_percentage_error(y_true[nonzero_mask], y_pred[nonzero_mask])
    else:
        mape = np.nan

    # SMAPE with zero guard
    numerator = np.abs(y_pred - y_true)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    valid = denominator > 0
    if valid.sum() > 0:
        smape = np.mean(numerator[valid] / denominator[valid])
    else:
        smape = np.nan

    naive_mae = mean_absolute_error(y_true, y_naive)
    mase = mae / naive_mae if naive_mae > 0 else np.nan

    forecast_bias = np.mean(y_pred - y_true)

    return {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'MAPE': mape,
            'SMAPE': smape, 'MASE': mase, 'Bias': forecast_bias}


# --- Bacon Bap detailed report ---
target_product = "Bacon Bap"
target_test_df = test_df[test_df['Product_Name'] == target_product].copy()
target_test_df = target_test_df.sort_values('Date').reset_index(drop=True)
target_test_df['Date_Only'] = target_test_df['Date'].dt.date

target_date_1 = pd.to_datetime('2025-11-02').date()
target_date_7 = pd.to_datetime('2025-11-08').date()
target_date_30 = pd.to_datetime('2025-11-30').date()

df_1_day = target_test_df[target_test_df['Date_Only'] == target_date_1]
df_1_week = target_test_df[
    (target_test_df['Date_Only'] >= target_date_1) & (target_test_df['Date_Only'] <= target_date_7)]
df_1_month = target_test_df[
    (target_test_df['Date_Only'] >= target_date_1) & (target_test_df['Date_Only'] <= target_date_30)]

comparison_df_bacon = pd.DataFrame({
    'Metric': ['WAPE', 'MASE', 'MAE', 'Bias', 'MSE', 'RMSE', 'MAPE', 'SMAPE'],
    '1-Day (Nov 2)': list(calculate_north_star_metrics(df_1_day).values()),
    '1-Week (Nov 2-8)': list(calculate_north_star_metrics(df_1_week).values()),
    '1-Month (Nov 2-30)': list(calculate_north_star_metrics(df_1_month).values())
})

for col in comparison_df_bacon.columns[1:]:
    comparison_df_bacon[col] = comparison_df_bacon[col].apply(
        lambda x: f"{float(x):.4f}" if not np.isnan(x) else "N/A"
    )

print("\n======================================================")
print("  RECURSIVE XGBOOST METRICS (BLIND TEST): BACON BAP")
print("======================================================")
print(comparison_df_bacon.to_string(index=False))

target_test_df['XGB_Prediction_Rounded'] = target_test_df['XGB_Prediction'].round().astype(int)
target_test_df['Mistake_Amount'] = (target_test_df['Sales'] - target_test_df['XGB_Prediction_Rounded']).abs()

print("\n==================================================")
print("  RECURSIVE XGBOOST REALITY CHECK: Bacon Bap")
print("==================================================")
print(target_test_df[['Date_Only', 'Sales', 'XGB_Prediction_Rounded', 'Mistake_Amount']].to_string(index=False))

# --- Aggregate metrics across ALL products ---
print("\n==================================================")
print("  PER-PRODUCT 1-MONTH METRICS (Nov 2-30)")
print("==================================================")

product_results = []
for product in all_products:
    p_df = test_df[test_df['Product_Name'] == product].copy()
    p_df['Date_Only'] = p_df['Date'].dt.date
    p_month = p_df[(p_df['Date_Only'] >= target_date_1) & (p_df['Date_Only'] <= target_date_30)]
    metrics = calculate_north_star_metrics(p_month)
    metrics['Product'] = product
    product_results.append(metrics)

product_leaderboard_df = pd.DataFrame(product_results)
product_leaderboard_df = product_leaderboard_df[['Product', 'MAE', 'RMSE', 'MAPE', 'MASE', 'Bias']]
product_leaderboard_df = product_leaderboard_df.sort_values('MAE')

for col in ['MAE', 'RMSE', 'MAPE', 'MASE', 'Bias']:
    product_leaderboard_df[col] = product_leaderboard_df[col].apply(
        lambda x: f"{x:.4f}" if not np.isnan(x) else "N/A"
    )

print(product_leaderboard_df.to_string(index=False))

# --- Overall aggregate ---
all_month = test_df.copy()
all_month['Date_Only'] = all_month['Date'].dt.date
all_month = all_month[(all_month['Date_Only'] >= target_date_1) & (all_month['Date_Only'] <= target_date_30)]
overall = calculate_north_star_metrics(all_month)

print("\n==================================================")
print("  OVERALL AGGREGATE METRICS (All Products, Nov 2-30)")
print("==================================================")
for k, v in overall.items():
    print(f"  {k:>6s}: {v:.4f}" if not np.isnan(v) else f"  {k:>6s}: N/A")

# --- Feature importance ---
print("\n==================================================")
print("  TOP 15 FEATURE IMPORTANCES (gain)")
print("==================================================")
importance = model.get_booster().get_score(importance_type='gain')
importance_sorted = sorted(importance.items(), key=lambda x: x[1], reverse=True)[:15]
for feat, score in importance_sorted:
    print(f"  {feat:<30s} {score:.1f}")


# ==========================================


  RECURSIVE XGBOOST METRICS (BLIND TEST): BACON BAP
Metric 1-Day (Nov 2) 1-Week (Nov 2-8) 1-Month (Nov 2-30)
  WAPE        0.3868           0.2446             0.3260
  MASE           N/A           1.2540             1.1165
   MAE       10.0579           4.2637             4.7769
  Bias      -10.0579          -1.6502             2.1505
   MSE      101.1611          30.0321            33.7786
  RMSE       10.0579           5.4802             5.8119
  MAPE        0.3868           0.2770             0.4206
 SMAPE        0.4796           0.2602             0.3218

  RECURSIVE XGBOOST REALITY CHECK: Bacon Bap
 Date_Only  Sales  XGB_Prediction_Rounded  Mistake_Amount
2025-11-02     26                      16              10
2025-11-03      8                      15               7
2025-11-04     14                      15               1
2025-11-05     14                      15               1
2025-11-06     22                      15               7
2025-11-07     18                      1

In [6]:
# ==========================================
# SAVE RESULTS — SQLite Storage Framework
# ==========================================
# I'm transitioning from flat CSVs to a proper two-table relational schema.
# This lets me track model performance over time and compare models in Tableau/PowerBI.

import sqlite3
from datetime import datetime

os.makedirs('../results', exist_ok=True)

# --- Generate a unique run ID so I can track this specific model run ---
prediction_timestamp = datetime.now()
run_id = prediction_timestamp.strftime('%Y%m%d_%H%M') + '_XGBoost_Improved_Daily'

print(f"Saving results for run: {run_id}")

# --- 1. Still save CSVs for quick access ---
comparison_df_bacon.to_csv('../results/xgboost_improved_daily_recursive_results.csv', index=False)
product_leaderboard_df.to_csv('../results/xgboost_improved_daily_per_product_results.csv', index=False)
test_df[['Date', 'Product_Name', 'Sales', 'XGB_Prediction']].to_csv(
    '../results/xgboost_improved_daily_all_predictions.csv', index=False)
print("  CSVs saved to ../results/")

# --- 2. SQLite Storage Framework ---
# Connect to (or create) my model tracking database
database_path = '../results/model_tracking.db'
db_connection = sqlite3.connect(database_path)

with db_connection:
    # Create tables if they don't exist yet
    # predictions_log: stores every single prediction for charting actual vs predicted
    db_connection.execute("""
        CREATE TABLE IF NOT EXISTS predictions_log (
            run_id TEXT,
            model_type TEXT,
            target_date TEXT,
            prediction_made_on TEXT,
            product_name TEXT,
            actual_sales REAL,
            predicted_sales REAL
        )
    """)

    # metrics_summary: stores aggregated North Star metrics per run/product/horizon
    db_connection.execute("""
        CREATE TABLE IF NOT EXISTS metrics_summary (
            run_id TEXT,
            model_type TEXT,
            product_name TEXT,
            evaluation_horizon TEXT,
            WAPE REAL,
            MASE REAL,
            MAE REAL,
            Bias REAL
        )
    """)

    # --- Populate predictions_log ---
    predictions_for_db = test_df[['Date', 'Product_Name', 'Sales', 'XGB_Prediction']].copy()
    predictions_for_db = predictions_for_db.rename(columns={
        'Date': 'target_date', 'Product_Name': 'product_name',
        'Sales': 'actual_sales', 'XGB_Prediction': 'predicted_sales'
    })
    predictions_for_db['run_id'] = run_id
    predictions_for_db['model_type'] = 'XGBoost_Improved_Daily'
    predictions_for_db['prediction_made_on'] = str(prediction_timestamp)
    predictions_for_db['target_date'] = predictions_for_db['target_date'].astype(str)
    predictions_for_db.to_sql('predictions_log', db_connection, if_exists='append', index=False)
    print(f"  Logged {len(predictions_for_db)} predictions to predictions_log")

    # --- Populate metrics_summary ---
    evaluation_start = pd.to_datetime('2025-11-02').date()
    evaluation_week_end = pd.to_datetime('2025-11-08').date()
    evaluation_month_end = pd.to_datetime('2025-11-30').date()

    test_df['Date_Only'] = test_df['Date'].dt.date
    month_data = test_df[(test_df['Date_Only'] >= evaluation_start) & (test_df['Date_Only'] <= evaluation_month_end)]
    week_data = month_data[month_data['Date_Only'] <= evaluation_week_end]
    day_data = month_data[month_data['Date_Only'] == evaluation_start]
    horizons = {'1-Day': day_data, '1-Week': week_data, '1-Month': month_data}

    metrics_rows = []
    for product in all_products:
        for horizon_label, horizon_df in horizons.items():
            product_horizon = horizon_df[horizon_df['Product_Name'] == product]
            if len(product_horizon) > 0:
                m = calculate_north_star_metrics(product_horizon)
                metrics_rows.append({
                    'run_id': run_id, 'model_type': 'XGBoost_Improved_Daily',
                    'product_name': product, 'evaluation_horizon': horizon_label,
                    'WAPE': m['WAPE'], 'MASE': m['MASE'], 'MAE': m['MAE'], 'Bias': m['Bias']
                })

    # ALL_PRODUCTS aggregate row
    for horizon_label, horizon_df in horizons.items():
        if len(horizon_df) > 0:
            m = calculate_north_star_metrics(horizon_df)
            metrics_rows.append({
                'run_id': run_id, 'model_type': 'XGBoost_Improved_Daily',
                'product_name': 'ALL_PRODUCTS', 'evaluation_horizon': horizon_label,
                'WAPE': m['WAPE'], 'MASE': m['MASE'], 'MAE': m['MAE'], 'Bias': m['Bias']
            })

    if metrics_rows:
        metrics_summary_df = pd.DataFrame(metrics_rows)
        metrics_summary_df.to_sql('metrics_summary', db_connection, if_exists='append', index=False)
        print(f"  Logged {len(metrics_summary_df)} metric rows to metrics_summary")

db_connection.close()
print(f"  SQLite database saved to {database_path}")
print(f"  Run ID: {run_id}")
print("\nDone!")


Saving results for run: 20260419_1540_XGBoost_Improved_Daily
  CSVs saved to ../results/
  Logged 1856 predictions to predictions_log
  Logged 195 metric rows to metrics_summary
  SQLite database saved to ../results/model_tracking.db
  Run ID: 20260419_1540_XGBoost_Improved_Daily

Done!
